# Checkpoint 4 — channel-grouped holdout and cross-validation

This notebook creates split assignments independently for the 7, 14, 21, and 30-day datasets.

- About 20% of rows are reserved as a test partition by channel.
- The remaining development rows receive one of five grouped validation folds.
- A channel can never appear on both sides of a holdout or fold.
- video_id and channel_id are saved only in separate split metadata; they are not model features.

**Evaluation note:** earlier feature exploration used all available rows. Therefore this test partition is protected from model fitting, but it was indirectly exposed during feature-selection decisions. A future untouched batch is preferable for the final unbiased product estimate.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

HORIZONS = (7, 14, 21, 30)
RANDOM_STATE = 42
TEST_SIZE = 0.20
HOLDOUT_CANDIDATES = 200
CV_FOLDS = 5

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

MASTER_PATH = PROJECT_ROOT / 'Dataset' / 'viewcastlk_training_table.csv'
MODEL_DATA_DIR = PROJECT_ROOT / 'Dataset' / 'model_horizon_datasets'
SPLIT_METADATA_DIR = PROJECT_ROOT / 'Dataset' / 'model_split_metadata'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'model_splits'
SPLIT_METADATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

master = pd.read_csv(MASTER_PATH, low_memory=False)
horizon_frames = {
    horizon: pd.read_csv(MODEL_DATA_DIR / f'viewcastlk_day_{horizon}.csv', low_memory=False)
    for horizon in HORIZONS
}

print(f'Loaded master dataset: {len(master):,} rows')
display(pd.DataFrame({
    'horizon_days': list(HORIZONS),
    'model_rows': [len(horizon_frames[h]) for h in HORIZONS],
}))

Loaded master dataset: 64,515 rows
   horizon_days  model_rows
0             7       20663
1            14       15685
2            21       15100
3            30       14753


In [2]:
def is_true(series):
    return series.astype(str).str.strip().str.lower().isin({'true', '1', 'yes'})


def horizon_source_mask(frame, horizon):
    target = f'd{horizon}_views'
    return (
        is_true(frame['eligible'])
        & is_true(frame[f'd{horizon}_usable'])
        & pd.to_numeric(frame[target], errors='coerce').notna()
    )


def choose_channel_holdout(groups):
    # Generate group-only candidates and select the one closest to 20% of rows.
    # No target values are used to choose the split.
    positions = np.arange(len(groups))
    splitter = GroupShuffleSplit(
        n_splits=HOLDOUT_CANDIDATES,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    candidates = list(splitter.split(positions, groups=groups))
    return min(candidates, key=lambda pair: abs(len(pair[1]) / len(groups) - TEST_SIZE))


def validation_fold_numbers(development_groups):
    fold_numbers = np.zeros(len(development_groups), dtype=np.int64)
    splitter = GroupKFold(n_splits=CV_FOLDS)
    positions = np.arange(len(development_groups))
    for fold, (_, validation_positions) in enumerate(
        splitter.split(positions, groups=development_groups), start=1
    ):
        fold_numbers[validation_positions] = fold
    return fold_numbers


split_summaries = []
fold_summaries = []
all_assignments = []

for horizon in HORIZONS:
    target = f'd{horizon}_views'
    source_mask = horizon_source_mask(master, horizon)
    source_rows = master.loc[source_mask, ['video_id', 'channel_id']].copy()
    source_rows.insert(0, 'source_row_index', source_rows.index)
    source_rows = source_rows.reset_index(drop=True)
    model_frame = horizon_frames[horizon]

    assert len(source_rows) == len(model_frame), (
        f'Day {horizon}: master/model row count mismatch. Rerun notebook 01 first.'
    )
    expected_target = pd.to_numeric(master.loc[source_mask, target], errors='coerce').reset_index(drop=True)
    actual_target = pd.to_numeric(model_frame[target], errors='coerce').reset_index(drop=True)
    pd.testing.assert_series_equal(actual_target, expected_target, check_names=False)

    groups = source_rows['channel_id'].astype(str).reset_index(drop=True)
    development_positions, test_positions = choose_channel_holdout(groups)

    assignments = source_rows.copy()
    assignments.insert(0, 'horizon_row_position', np.arange(len(assignments)))
    assignments.insert(0, 'horizon_days', horizon)
    assignments['partition'] = 'development'
    assignments.loc[test_positions, 'partition'] = 'test_reserved'
    assignments['cv_validation_fold'] = pd.Series(pd.NA, index=assignments.index, dtype='Int64')

    development_groups = assignments.loc[development_positions, 'channel_id'].astype(str).reset_index(drop=True)
    fold_numbers = validation_fold_numbers(development_groups)
    assignments.loc[development_positions, 'cv_validation_fold'] = fold_numbers

    output_path = SPLIT_METADATA_DIR / f'viewcastlk_day_{horizon}_split_assignments.csv'
    assignments.to_csv(output_path, index=False)
    all_assignments.append(assignments)

    development = assignments[assignments['partition'] == 'development']
    test = assignments[assignments['partition'] == 'test_reserved']
    development_channels = set(development['channel_id'].astype(str))
    test_channels = set(test['channel_id'].astype(str))

    split_summaries.append({
        'horizon_days': horizon,
        'total_rows': len(assignments),
        'development_rows': len(development),
        'test_rows': len(test),
        'test_row_percent': round(100 * len(test) / len(assignments), 2),
        'development_channels': len(development_channels),
        'test_channels': len(test_channels),
        'channel_overlap': len(development_channels & test_channels),
    })

    for fold in range(1, CV_FOLDS + 1):
        validation = development[development['cv_validation_fold'] == fold]
        training = development[development['cv_validation_fold'] != fold]
        training_channels = set(training['channel_id'].astype(str))
        validation_channels = set(validation['channel_id'].astype(str))
        fold_summaries.append({
            'horizon_days': horizon,
            'fold': fold,
            'training_rows': len(training),
            'validation_rows': len(validation),
            'training_channels': len(training_channels),
            'validation_channels': len(validation_channels),
            'channel_overlap': len(training_channels & validation_channels),
        })

split_summary = pd.DataFrame(split_summaries)
fold_summary = pd.DataFrame(fold_summaries)
combined_assignments = pd.concat(all_assignments, ignore_index=True)

combined_assignments.to_csv(SPLIT_METADATA_DIR / 'all_horizon_split_assignments.csv', index=False)
split_summary.to_csv(RESULTS_DIR / 'split_summary.csv', index=False)
fold_summary.to_csv(RESULTS_DIR / 'cv_fold_summary.csv', index=False)

print('Holdout summary')
display(split_summary)
print('Cross-validation fold summary')
display(fold_summary)

Holdout summary
   horizon_days  total_rows  ...  test_channels  channel_overlap
0             7       20663  ...            351                0
1            14       15685  ...            338                0
2            21       15100  ...            326                0
3            30       14753  ...            330                0

[4 rows x 8 columns]
Cross-validation fold summary
    horizon_days  fold  ...  validation_channels  channel_overlap
0              7     1  ...                  230                0
1              7     2  ...                  287                0
2              7     3  ...                  295                0
3              7     4  ...                  294                0
4              7     5  ...                  295                0
5             14     1  ...                  238                0
6             14     2  ...                  271                0
7             14     3  ...                  280                0
8            

In [3]:
test_results = []


def check(horizon, name, condition, detail=''):
    test_results.append({
        'horizon_days': horizon,
        'test': name,
        'status': 'PASS' if bool(condition) else 'FAIL',
        'detail': detail,
    })


for horizon in HORIZONS:
    model_frame = horizon_frames[horizon]
    assignments = pd.read_csv(
        SPLIT_METADATA_DIR / f'viewcastlk_day_{horizon}_split_assignments.csv'
    )
    development = assignments[assignments['partition'] == 'development']
    test = assignments[assignments['partition'] == 'test_reserved']
    development_channels = set(development['channel_id'].astype(str))
    test_channels = set(test['channel_id'].astype(str))
    observed_folds = set(development['cv_validation_fold'].dropna().astype(int))

    check(horizon, 'one assignment per model row', len(assignments) == len(model_frame))
    check(
        horizon,
        'row positions are complete and ordered',
        assignments['horizon_row_position'].tolist() == list(range(len(model_frame))),
    )
    check(horizon, 'video_id absent from model features', 'video_id' not in model_frame.columns)
    check(horizon, 'channel_id absent from model features', 'channel_id' not in model_frame.columns)
    check(
        horizon,
        'holdout has zero channel overlap',
        development_channels.isdisjoint(test_channels),
        f'overlap={len(development_channels & test_channels)}',
    )
    check(
        horizon,
        'every row has one valid partition',
        assignments['partition'].isin({'development', 'test_reserved'}).all(),
    )
    check(horizon, 'test rows have no CV fold', test['cv_validation_fold'].isna().all())
    check(horizon, 'development rows have a CV fold', development['cv_validation_fold'].notna().all())
    check(horizon, 'all five folds are represented', observed_folds == set(range(1, CV_FOLDS + 1)))
    test_share = len(test) / len(assignments)
    check(
        horizon,
        'test row share is within one percentage point of 20%',
        abs(test_share - TEST_SIZE) <= 0.01,
        f'test_share={test_share:.4f}',
    )

    for fold in range(1, CV_FOLDS + 1):
        validation = development[development['cv_validation_fold'] == fold]
        training = development[development['cv_validation_fold'] != fold]
        training_channels = set(training['channel_id'].astype(str))
        validation_channels = set(validation['channel_id'].astype(str))
        check(
            horizon,
            f'fold {fold} has zero channel overlap',
            training_channels.isdisjoint(validation_channels),
            f'overlap={len(training_channels & validation_channels)}',
        )

test_results = pd.DataFrame(test_results)
display(test_results)

failures = test_results[test_results['status'] == 'FAIL']
assert failures.empty, f'Split validation failed:\n{failures.to_string(index=False)}'
print(f'PASS: all {len(test_results)} split checks succeeded.')

    horizon_days  ...             detail
0              7  ...                   
1              7  ...                   
2              7  ...                   
3              7  ...                   
4              7  ...          overlap=0
5              7  ...                   
6              7  ...                   
7              7  ...                   
8              7  ...                   
9              7  ...  test_share=0.1991
10             7  ...          overlap=0
11             7  ...          overlap=0
12             7  ...          overlap=0
13             7  ...          overlap=0
14             7  ...          overlap=0
15            14  ...                   
16            14  ...                   
17            14  ...                   
18            14  ...                   
19            14  ...          overlap=0
20            14  ...                   
21            14  ...                   
22            14  ...                   
23            14

## Checkpoint result

The saved assignment files are the single source of truth for the reserved test rows and five development folds. Join them to a horizon dataset by horizon_row_position; do not add video_id or channel_id to the model feature matrix.

The next checkpoint is fold-safe preprocessing: learn imputers and encoders only from each training fold, then apply them to its validation fold.